In [47]:
# pandasをpdという名前でインポート
import pandas as pd

In [48]:
# CSVファイルを読み込みデータフレームを作成
df = pd.read_csv('KvsT.csv')

# データフレームの先頭10行を表示 10行にしているのは、性別列が男性/女性の両方のデータを表示したいため
df.head(10)

,身長,体重,年代,派閥
0,170,60,10,きのこ
1,172,65,20,きのこ
2,170,60,30,たけのこ
3,170,65,40,きのこ
4,177,65,10,たけのこ
5,168,55,20,きのこ
6,169,65,30,たけのこ
7,170,62,40,たけのこ
8,180,70,10,たけのこ
9,170,68,20,きのこ


In [49]:
df.shape

(19, 4)

In [50]:
df.columns

Index(['身長', '体重', '年代', '派閥'], dtype='object')

In [51]:
df.dtypes

身長     int64
体重     int64
年代     int64
派閥    object
dtype: object

In [52]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   身長      19 non-null     int64 
 1   体重      19 non-null     int64 
 2   年代      19 non-null     int64 
 3   派閥      19 non-null     object
dtypes: int64(3), object(1)
memory usage: 736.0+ bytes


In [53]:
df.describe()

,身長,体重,年代
count,19.000000,19.000000,19.000000
mean,169.789474,64.000000,25.789474
std,5.563561,5.547772,10.706068
min,155.000000,55.000000,10.000000
25%,169.500000,60.000000,20.000000
50%,170.000000,65.000000,20.000000
75%,171.000000,65.000000,35.000000
max,180.000000,77.000000,40.000000


In [54]:
# データフレームの特定の列を取り出す df['列名']
df['派閥'].value_counts()

派閥
たけのこ    11
きのこ      8
Name: count, dtype: int64

In [55]:
# 変換用の辞書を作る
gender_mapper = {'たけのこ': 0, 'きのこ': 1}

# データフレームから特定の列を取り出し、変換用の辞書で変換
# 変換結果は'gender_encoded'という列名でデータフレームに追加
df['派閥_encoded'] = df['派閥'].replace(gender_mapper)

# データフレームの先頭10行を表示
df.head(10)

/tmp/ipykernel_10377/3437991884.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['派閥_encoded'] = df['派閥'].replace(gender_mapper)


,身長,体重,年代,派閥,派閥_encoded
0,170,60,10,きのこ,1
1,172,65,20,きのこ,1
2,170,60,30,たけのこ,0
3,170,65,40,きのこ,1
4,177,65,10,たけのこ,0
5,168,55,20,きのこ,1
6,169,65,30,たけのこ,0
7,170,62,40,たけのこ,0
8,180,70,10,たけのこ,0
9,170,68,20,きのこ,1


In [56]:
# '年代'列のカテゴリカルデータをワンホットエンコーディングする
# columns:ワンホットエンコーディングをする列のリスト
# drop_first:True ワンホットエンコーディングした最初の列を削除する
df = pd.get_dummies(df, columns=['年代'], drop_first=True,dtype=int)

# データフレームの先頭10行を表示
df.head(10)

,身長,体重,派閥,派閥_encoded,年代_20,年代_30,年代_40
0,170,60,きのこ,1,0,0,0
1,172,65,きのこ,1,1,0,0
2,170,60,たけのこ,0,0,1,0
3,170,65,きのこ,1,0,0,1
4,177,65,たけのこ,0,0,0,0
5,168,55,きのこ,1,1,0,0
6,169,65,たけのこ,0,0,1,0
7,170,62,たけのこ,0,0,0,1
8,180,70,たけのこ,0,0,0,0
9,170,68,きのこ,1,1,0,0


In [57]:
# 年代をワンホットエンコーディングした結果の列 → '年代_20','年代_30','年代_40'
# genderをエンコーディングした結果の列 → '派閥_encoded'
# モデルの学習に使うためには、目的変数(正解ラベル)の列も必要 → '派閥_encoded'

# XGBoostの場合は先頭に目的変数(正解ラベル)、それ以降を特徴量とする
df_for_use = df[ ['派閥_encoded', 
                  '身長',
                  '体重',
                  '年代_20',
                  '年代_30',
                  '年代_40'] ].copy()

# モデルの学習で使うデータの先頭5行を表示
df_for_use.head()

,派閥_encoded,身長,体重,年代_20,年代_30,年代_40
0,1,170,60,0,0,0
1,1,172,65,1,0,0
2,0,170,60,0,1,0
3,1,170,65,0,0,1
4,0,177,65,0,0,0


In [58]:
# 機械学習用ライブラリのsklearnのmode_selectionモジュールからデータ分割関数のtrain_test_split関数をインポート
from sklearn.model_selection import train_test_split

In [59]:
# 分割1回目
# データセットを トレーニング用(train) と テスト＆検証用(train_and_validate) に分割する
# test_size:テスト＆検証用の割合 検証用1割とテスト用1割の合計なので2割(0.2)を指定
# random_state:乱数シード 任意の値でOK
# stratify:データフレームdf_for_useの目的変数(正解ラベル)の列のばらつき具合を維持して分割する
train, test_and_validate = train_test_split(df_for_use, test_size=0.2, random_state=42, stratify=df['派閥_encoded'])

In [60]:
# うまく分割できているか確認
# トレーニングデータセットの行数と列数を表示
print(train.shape)
# トレーニングデータセットの正解ラベルのデータの種類と件数を表示
print(train['派閥_encoded'].value_counts())

# テスト＆検証用データセットの行数と列数を表示
print(test_and_validate.shape)
# テスト＆検証用データセットの正解ラベルのデータの種類と件数を表示
print(test_and_validate['派閥_encoded'].value_counts())

(15, 6)
派閥_encoded
0    9
1    6
Name: count, dtype: int64
(4, 6)
派閥_encoded
0    2
1    2
Name: count, dtype: int64


In [61]:
# 分割2回目
# テスト&検証用データセットを テスト用(test) と 検証用(validate) に分割する
# test_size:半分ずつなので5割(0.5)を指定
# random_state:乱数シード 任意の値でOK
# stratify:データフレームtest_and_validateの目的変数(正解ラベル)の列のばらつき具合を維持して分割する
test, validate = train_test_split(test_and_validate, test_size=0.5, random_state=42, stratify=test_and_validate['派閥_encoded'])

In [62]:
# うまく分割できているか確認
# テストデータセットの行数と列数を表示
print(test.shape)
# テストデータセットの正解ラベルのデータの種類と件数を表示
print(test['派閥_encoded'].value_counts())

# 検証用データセットの行数と列数を表示
print(validate.shape)
# 検証用データセットの正解ラベルのデータの種類と件数を表示
print(validate['派閥_encoded'].value_counts())

(2, 6)
派閥_encoded
1    1
0    1
Name: count, dtype: int64
(2, 6)
派閥_encoded
1    1
0    1
Name: count, dtype: int64


In [63]:
# 関数の中で「io」を使うため、importし忘れに注意。
import io

In [64]:
# S3バケットとプレフィックスの設定
#バケット名は必ずSandboxのものを使う
# prefixは任意でOK(コピペすればlab3)
bucket='c167743a4309710l10786217t1w968768184-sandboxbucket-w6fhkw804evp'
prefix='lab3'

# トレーニングデータ、テストデータ、検証データのファイル名設定 任意でOK
train_file='train.csv'
test_file='test.csv'
validate_file='validate.csv'

# osモジュールのインポート(ファイルパスの生成用)
import os

# S3へアクセスする準備
# boto3セッションからS3リソースを取得
import boto3
s3_resource = boto3.Session().resource('s3')

# S3にCSVファイルをアップロードするための関数を定義
def upload_s3_csv(filename, folder, dataframe):
    csv_buffer = io.StringIO()
    dataframe.to_csv(csv_buffer, header=False, index=False)
    # ★ 以下の部分でS3にデータ(dataframeの内容)をアップしている
    s3_resource.Bucket(bucket).Object(os.path.join(prefix, folder, filename)).put(Body=csv_buffer.getvalue())

INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


In [65]:
# DataFrameをS3にアップロードする upload_s3_csv関数を使用

# ファイル名:train_fileの値
# フォルダ名:'train'
# アップするデータ:train(DataFrame)の内容
upload_s3_csv(train_file, 'train', train)

# ファイル名:test_fileの値
# フォルダ名:'test'
# アップするデータ:test(DataFrame)の内容
upload_s3_csv(test_file, 'test', test)

# ファイル名:validate_fileの値
# フォルダ名:'validate'
# アップするデータ:validate(DataFrame)の内容
upload_s3_csv(validate_file, 'validate', validate)

In [66]:
# Amazon SageMaker用のモジュールをインポート
import boto3
from sagemaker.image_uris import retrieve

# SageMakerにおいてXGBoostアルゴリズムを利用するためのDockerコンテナイメージを取得
container = retrieve('xgboost',boto3.Session().region_name,'1.0-1')

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3
INFO:sagemaker.image_uris:Defaulting to only supported image scope: cpu.


In [67]:
# XGBoostのハイパーパラメータを辞書で保持
# num_round: 学習回数 大きすぎると過学習、小さすぎると学習不足
# eval_metric:モデルの評価指標 'auc'...ROC曲線の面積 0～1の範囲で1に近いほど分類性能が高い
# objective:目的関数(この関数の計算結果を最小/最大にするように学習する) 'binary:logistic'...2クラス分類のための関数
hyperparams={"num_round":"42",
             "eval_metric": "auc",
             "objective": "binary:logistic"}

In [68]:
# SageMaker SDKを使用して、XGBoostのモデルをトレーニングおよびデプロイするために必要なモジュールをインポート
import sagemaker

# トレーニング結果を保存するS3のパス
s3_output_location="s3://{}/{}/output/".format(bucket, prefix)

# SageMakerのXGBoostアルゴリズムを使用するためにXGBoost Estimatorを作成
# image_uri:使用するコンテナ
# role:SageMakerの実行ロール sagemaker.get_execution_role()で取得した値
# instance_count:トレーニングに使用するインスタンスの数
# instance_type:トレーニングに使用するインスタンスのタイプ
# output_path: トレーニング結果を保存するパス
# hyperparameters: ハイパーパラメータ
# sagemaker_session: SageMakerのセッション sagemaker.session()で取得した値
xgb_model=sagemaker.estimator.Estimator(image_uri=container,
                                       role=sagemaker.get_execution_role(),
                                       instance_count=1,
                                       instance_type='ml.m4.xlarge',
                                       output_path=s3_output_location,
                                        hyperparameters=hyperparams,
                                        sagemaker_session=sagemaker.Session())

In [69]:
# SageMakerのトレーニングジョブに使用するトレーニングデータのパスとファイルの種類'text/csv'を指定

# トレーニング用データオブジェクトを作る (S3から指定したCSVを読み込む)
train_channel = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/train/".format(bucket, prefix, train_file), # パス(～/train/～)は、S3にアップする時に設定していたもの
    content_type='text/csv')

# 検証用データオブジェクトを作る (S3から指定したCSVを読み込む)
validate_channel = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/validate/".format(bucket, prefix, validate_file), # パス(～/validate/～)は、S3にアップする時に設定していたもの
    content_type='text/csv')

# トレーニングジョブで使うデータを辞書型で保持しておく（学習の時にこれで指定する）
# キー'train'にトレーニング用データオブジェクト、キー'validation'に検証用データオブジェクト を指定
data_channels = {'train': train_channel, 'validation': validate_channel}

In [70]:
xgb_model.fit(inputs=data_channels, logs=False)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2025-06-30-08-43-52-871



2025-06-30 08:43:53 Starting - Starting the training job....
2025-06-30 08:44:20 Starting - Preparing the instances for training....
2025-06-30 08:44:47 Downloading - Downloading input data.....
2025-06-30 08:45:17 Downloading - Downloading the training image.........
2025-06-30 08:46:08 Training - Training image download completed. Training in progress.....
2025-06-30 08:46:30 Uploading - Uploading generated training model.
2025-06-30 08:46:43 Completed - Training job completed


In [71]:
# ★ リアルタイム推論
# XGBoostモデルをデプロイしてエンドポイントを作成
# xgb_model.fit()が済んでいるモデルに対してdeployメソッドを呼び出し、エンドポイントを取得
xgb_predictor = xgb_model.deploy(initial_instance_count=1, # インスタンスの数を指定
                serializer = sagemaker.serializers.CSVSerializer(), # シリアライザをCSV形式に設定
                instance_type='ml.m4.xlarge') # インスタンスタイプを指定

INFO:sagemaker:Creating model with name: sagemaker-xgboost-2025-06-30-08-46-44-942
INFO:sagemaker:Creating endpoint-config with name sagemaker-xgboost-2025-06-30-08-46-44-942
INFO:sagemaker:Creating endpoint with name sagemaker-xgboost-2025-06-30-08-46-44-942


---------!

In [72]:
# 再確認としてテスト用データ(test)の中身を見てみる
test.head()

# このデータの1行目 かつ age列より後ろの列を、推論用のデータとして用いたい
# ※推論させるときは特徴量のみ使う。正解ラベルは渡さない。

,派閥_encoded,身長,体重,年代_20,年代_30,年代_40
17,1,170,65,0,0,1
12,0,175,77,0,0,1


In [73]:
# ilocを使い、行インデックス0の行かつ、列インデックス1～末尾のデータを抽出する
row = test.iloc[0:1,1:] # テストデータから行インデックス0で、列インデックス1以降のデータを取得

# 抽出した結果を表示
row.head()

,身長,体重,年代_20,年代_30,年代_40
17,170,65,0,0,1


In [74]:
# rowのデータを カンマ区切り値の文字列 にする
batch_X_csv_buffer = io.StringIO()
row.to_csv(batch_X_csv_buffer, header=False, index=False)
test_row = batch_X_csv_buffer.getvalue()

# 変換した文字列を表示
print(test_row)

170,65,0,0,1



In [75]:
# ★ モデルに予測させる エンドポイント.predictメソッド
xgb_predictor.predict(test_row)

# 予測した結果、1である確率が表示される

b'0.26134973764419556'